# Single-Note Pipeline Run With Context Agent

Run one MIMIC-III-Ext-Notes case through the full EHR pipeline with Stage 4 context agent enabled. This notebook is verbose, showing the workings of the context agent within the pipeline

In [1]:
import importlib
import json
import logging
import sys
import traceback
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

# Reload local modules so notebook edits to config/prompts/stages are picked up.
import ehr_pipeline.config
import ehr_pipeline.prompts
import ehr_pipeline.ollama_client
import ehr_pipeline.schemas
import ehr_pipeline.evidence_store
import ehr_pipeline.runtime
import ehr_pipeline.stages.s1_evidence
import ehr_pipeline.stages.s2_extract
import ehr_pipeline.stages.s3_verify
import ehr_pipeline.stages.s4_context
import ehr_pipeline.stages.s5_fact_sheet
import ehr_pipeline.stages.s6_summarize
import ehr_pipeline.stages.s7_check
import ehr_pipeline.stages.s8_review
import ehr_pipeline.stages.s9_patient_summary
import ehr_pipeline.extraction
import ehr_pipeline.verification
import ehr_pipeline.pipeline
import benchmarks.mimic
import benchmarks.metrics

for _mod in [
    ehr_pipeline.config,
    ehr_pipeline.prompts,
    ehr_pipeline.ollama_client,
    ehr_pipeline.schemas,
    ehr_pipeline.evidence_store,
    ehr_pipeline.runtime,
    ehr_pipeline.stages.s1_evidence,
    ehr_pipeline.stages.s2_extract,
    ehr_pipeline.stages.s3_verify,
    ehr_pipeline.stages.s4_context,
    ehr_pipeline.stages.s5_fact_sheet,
    ehr_pipeline.stages.s6_summarize,
    ehr_pipeline.stages.s7_check,
    ehr_pipeline.stages.s8_review,
    ehr_pipeline.stages.s9_patient_summary,
    ehr_pipeline.extraction,
    ehr_pipeline.verification,
    ehr_pipeline.pipeline,
    benchmarks.mimic,
    benchmarks.metrics,
]:
    importlib.reload(_mod)

from ehr_pipeline import config as pipeline_config
from ehr_pipeline.ollama_client import OllamaError
from ehr_pipeline.pipeline import run_pipeline
from benchmarks import mimic, metrics

print("project root:", PROJECT_ROOT)
print("ollama host:", pipeline_config.OLLAMA_HOST)
print("api key set:", bool(pipeline_config.OLLAMA_API_KEY))
print("models:")
for k, v in vars(pipeline_config.MODELS).items():
    print(f"  {k}: {v}")

project root: /Users/natejly/Documents/GitHub/LLMS
ollama host: https://ollama.com
api key set: True
models:
  claim_extraction: gemma4:31b
  claim_verification: gemma4:31b
  context_agent: gemma4:31b
  summary_generation: gemma4:31b
  final_review: gemma4:31b


In [2]:
# Dataset paths
EHR_JSON_PATH = PROJECT_ROOT / "datasets" / "bquxjob_5dbe3bd2_19dae30f3cf.json"
NOTES_CSV_PATH = PROJECT_ROOT / "datasets" / "mimic-iii-ext-notes-1.0.0" / "notes.csv"
LABELS_CSV_PATH = PROJECT_ROOT / "datasets" / "mimic-iii-ext-notes-1.0.0" / "labels.csv"

# Materialized benchmark inputs for this notebook only.
WORK_DIR = PROJECT_ROOT / "outputs" / "_single_context_agent"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Pick one case. Set CASE_ID to a specific mimic-* id if desired; otherwise CASE_INDEX is used.
CASE_ID = None
CASE_INDEX = 0

# Force a full fresh run for the debug case. Set RESUME=True if you want to reuse cached debug outputs.
RESUME = False
ALLOW_VIOLATIONS = True
ENABLE_CONTEXT_AGENT = True
SUMMARY_MIN_RATIO = 0.20
SUMMARY_MAX_RATIO = 0.30
BERTSCORE_MODEL = "roberta-large"

all_cases = mimic.load_real_cases(
    ehr_json_path=EHR_JSON_PATH,
    notes_csv_path=NOTES_CSV_PATH,
    labels_csv_path=LABELS_CSV_PATH,
)

if CASE_ID:
    case = next(c for c in all_cases if c.case_id == CASE_ID)
else:
    case = all_cases[CASE_INDEX]

RUN_CASE_ID = f"{case.case_id}-context-agent-debug"

print(f"loaded {len(all_cases)} cases")
print("selected case:", case.case_id)
print("debug run case_id:", RUN_CASE_ID)
print("hadm_id:", case.hadm_id, "subject_id:", case.subject_id, "row_id:", case.note_row_id)
print("note chars:", len(case.note_text), "gold concepts:", len(case.gold_concepts))
print("context agent:", "ON" if ENABLE_CONTEXT_AGENT else "OFF")
print("resume:", RESUME)

loaded 150 cases
selected case: mimic-81475-101662-673718
debug run case_id: mimic-81475-101662-673718-context-agent-debug
hadm_id: 101662 subject_id: 81475 row_id: 673718
note chars: 2295 gold concepts: 8
context agent: ON
resume: False


In [3]:
case_overview = pd.DataFrame([
    {
        "case_id": case.case_id,
        "debug_case_id": RUN_CASE_ID,
        "subject_id": case.subject_id,
        "hadm_id": case.hadm_id,
        "note_row_id": case.note_row_id,
        "note_chars": len(case.note_text),
        "gold_concepts": len(case.gold_concepts),
        "diagnoses": len(case.admission.get("diagnoses", [])),
        "procedures": len(case.admission.get("procedures", [])),
    }
])
case_overview

,case_id,debug_case_id,subject_id,hadm_id,note_row_id,note_chars,gold_concepts,diagnoses,procedures
0,mimic-81475-101662-673718,mimic-81475-101662-673718-context-agent-debug,81475,101662,673718,2295,8,0,0


In [4]:
print(case.note_text[:2500])

60 y/o M  w/ an extensive PMH including R Hip hemiarthroplasty,    diverticulitis, Renal transplant 01, who presented to ED complaining    of worsening SOB x 2 days.  In the ED his vitals were significant for a    NBP 193/113, and profound insp/exp wheezes, he received several    abluterol/atrovent nebs, and was transferred to the M/SICU for further    management of a COPD exacerbation and hypertension.    Hypertension, benign    Assessment:    SBP 110 s to 170 s with HR 110 s to 120 s throughout shift. Pt. remains    on fentanyl 100mcg/kg and propofol 90mcg/kg for sedation/comfort.    Action:    Pt. given IV hydralazine q6hr along with drips as noted above. Pt.    bolused with 100mcg of fentanyl prior to nsg. care.    Response:    SBP remained labile throughout shift ranging from 110 s to 170    Remained tachycardic throughout shift.    Plan:    Continue to monitor patient hemodynamic status closely. Continue with    propofol, fentanyl and hydralazine as ordered. If pt. remains    hyp

## Run Full Pipeline

This cell materializes the selected note and structured EHR as benchmark inputs, then runs all pipeline stages with the context agent enabled. With `RESUME = False`, it performs a fresh run for the debug case id.

In [5]:
bundle_path, notes_dir = mimic.materialize_real_case(case, WORK_DIR)
print("bundle:", bundle_path)
print("notes_dir:", notes_dir)
print("output_dir:", pipeline_config.output_dir(RUN_CASE_ID))

try:
    result = run_pipeline(
        case_id=RUN_CASE_ID,
        bundle_path=bundle_path,
        notes_dir=notes_dir,
        resume=RESUME,
        allow_violations=ALLOW_VIOLATIONS,
        enable_context_agent=ENABLE_CONTEXT_AGENT,
        summary_min_ratio=SUMMARY_MIN_RATIO,
        summary_max_ratio=SUMMARY_MAX_RATIO,
    )
except OllamaError as exc:
    print(f"Ollama error: {exc}")
    if exc.status_code == 401:
        print("Hint: local Ollama rejects Bearer tokens. Use OLLAMA_HOST=https://ollama.com or unset OLLAMA_API_KEY for local runs.")
    elif exc.status_code == 404:
        print("Hint: model not found. Check ehr_pipeline/config.py model assignments.")
    traceback.print_exc()
    raise

print("done:", result.case_id)
print("summary:", result.summary_path)
print("audit:", result.audit_path)

2026-05-03 17:19:12,188 INFO ehr_pipeline.pipeline: Saved input note (2295 chars) -> /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-81475-101662-673718-context-agent-debug/input_note.txt
2026-05-03 17:19:12,189 INFO ehr_pipeline.stages.s1_evidence: Stage 1: building evidence store for case mimic-81475-101662-673718-context-agent-debug
2026-05-03 17:19:12,191 INFO ehr_pipeline.stages.s1_evidence:   wrote 73 evidence items (41 structured, 32 note sentences) to /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-81475-101662-673718-context-agent-debug/evidence_store.json
2026-05-03 17:19:12,191 INFO ehr_pipeline.runtime:   stage1_evidence done in 0.00s
2026-05-03 17:19:12,192 INFO ehr_pipeline.stages.s2_extract: Stage 2: extracting claims from notes


bundle: /Users/natejly/Documents/GitHub/LLMS/outputs/_single_context_agent/mimic-81475-101662-673718/bundle.json
notes_dir: /Users/natejly/Documents/GitHub/LLMS/outputs/_single_context_agent/mimic-81475-101662-673718/notes
output_dir: /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-81475-101662-673718-context-agent-debug


2026-05-03 17:21:22,239 INFO ehr_pipeline.stages.s2_extract:   extracted 44 claims, wrote /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-81475-101662-673718-context-agent-debug/claims.json
2026-05-03 17:21:22,241 INFO ehr_pipeline.runtime:   stage2_extract done in 130.05s
2026-05-03 17:21:22,241 INFO ehr_pipeline.stages.s3_verify: Stage 3: verifying 44 claims
2026-05-03 17:22:07,105 INFO ehr_pipeline.stages.s3_verify:   {'verified': 32, 'contradicted': 0, 'unsupported': 12} -> /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-81475-101662-673718-context-agent-debug/verifications.json
2026-05-03 17:22:07,105 INFO ehr_pipeline.runtime:   stage3_verify done in 44.86s
2026-05-03 17:22:07,105 INFO ehr_pipeline.stages.s4_context: Stage 4: context agent
2026-05-03 17:22:20,583 INFO ehr_pipeline.stages.s4_context:   context agent proposed 4 new claims; re-verifying
2026-05-03 17:22:20,586 INFO ehr_pipeline.stages.s3_verify: Stage 3: verifying 4 claims
2026-05-03 17:22:27,876 INFO ehr_pipe

done: mimic-81475-101662-673718-context-agent-debug
summary: /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-81475-101662-673718-context-agent-debug/summary.md
audit: /Users/natejly/Documents/GitHub/LLMS/outputs/mimic-81475-101662-673718-context-agent-debug/audit.json


In [6]:
def read_json(path: Path):
    return json.loads(path.read_text("utf-8")) if path.exists() else None


def read_text(path: Path) -> str:
    return path.read_text("utf-8") if path.exists() else ""


out_dir = pipeline_config.output_dir(RUN_CASE_ID)
artifact_paths = {
    "input_note": out_dir / "input_note.txt",
    "evidence_store": out_dir / "evidence_store.json",
    "claims": out_dir / "claims.json",
    "verifications": out_dir / "verifications.json",
    "context": out_dir / "context.json",
    "claims_augmented": out_dir / "claims_augmented.json",
    "verifications_augmented": out_dir / "verifications_augmented.json",
    "fact_sheet": out_dir / "fact_sheet.json",
    "summary": out_dir / "summary.md",
    "summary_revised": out_dir / "summary_revised.md",
    "summary_meta": out_dir / "summary_meta.json",
    "check_report": out_dir / "check_report.json",
    "check_report_revised": out_dir / "check_report_revised.json",
    "review": out_dir / "review.json",
    "patient_summary": out_dir / "patient_summary.md",
    "patient_summary_meta": out_dir / "patient_summary_meta.json",
    "audit": out_dir / "audit.json",
}

pd.DataFrame([
    {"artifact": name, "exists": path.exists(), "path": str(path)}
    for name, path in artifact_paths.items()
])

,artifact,exists,path
0,input_note,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
1,evidence_store,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
2,claims,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
3,verifications,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
4,context,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
5,verifications_augmented,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
6,fact_sheet,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
7,summary,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
8,summary_revised,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...
9,summary_meta,True,/Users/natejly/Documents/GitHub/LLMS/outputs/m...


In [7]:
timings_df = pd.DataFrame([t.__dict__ for t in result.timings])
if not timings_df.empty:
    timings_df["seconds"] = timings_df["seconds"].round(2)
    display(timings_df)
    print(f"Total seconds: {timings_df['seconds'].sum():.2f}")
else:
    print("No timing data found.")

,name,seconds,skipped
0,stage1_evidence,0.00,False
1,stage2_extract,130.05,False
2,stage3_verify,44.86,False
3,stage4_context,20.77,False
4,stage5_fact_sheet,0.00,False
5,stage9_patient_summary,13.09,False
6,stage6_summarize,10.02,False
7,stage7_check,0.01,False
8,stage8_review,5.64,False


Total seconds: 224.44


In [8]:
claims = read_json(artifact_paths["claims"]) or {"claims": []}
verifications = read_json(artifact_paths["verifications"]) or {"verifications": []}
augmented = read_json(artifact_paths["verifications_augmented"]) or {"verifications": []}

claims_df = pd.DataFrame(claims.get("claims", []))
ver_df = pd.DataFrame(verifications.get("verifications", []))
aug_df = pd.DataFrame(augmented.get("verifications", []))

print("claims:", len(claims_df))
print("raw verifications:", len(ver_df))
print("augmented verifications:", len(aug_df))

if not ver_df.empty:
    display(ver_df["status"].value_counts().rename_axis("status").reset_index(name="raw_count"))
if not aug_df.empty:
    display(aug_df["status"].value_counts().rename_axis("status").reset_index(name="augmented_count"))

claims_df.head(10)

claims: 44
raw verifications: 44
augmented verifications: 48


,status,raw_count
0,verified,32
1,unsupported,12


,status,augmented_count
0,verified,36
1,unsupported,12


,claim_id,type,subject,predicate,value,time_ref,source_span
0,C1,procedure,patient,has history of,R Hip hemiarthroplasty,None,row673718#0
1,C2,diagnosis,patient,has diagnosis,diverticulitis,None,row673718#0
2,C3,procedure,patient,has history of,Renal transplant,01,row673718#0
3,C4,diagnosis,patient,complaining of,worsening SOB,x 2 days,row673718#0
4,C5,vital,patient,lab result,NBP 193/113,ED,row673718#1
5,C6,diagnosis,patient,has,profound insp/exp wheezes,ED,row673718#1
6,C7,medication,patient,received,abluterol/atrovent nebs,ED,row673718#1
7,C8,diagnosis,patient,has diagnosis,COPD exacerbation,None,row673718#1
8,C9,diagnosis,patient,has diagnosis,hypertension,None,row673718#1
9,C10,diagnosis,patient,has diagnosis,"Hypertension, benign",None,row673718#2


In [9]:
context = read_json(artifact_paths["context"]) or {}
augmented = read_json(artifact_paths["verifications_augmented"]) or {"verifications": []}
fact_sheet = read_json(artifact_paths["fact_sheet"]) or {"sections": {}}
claims_augmented = read_json(artifact_paths["claims_augmented"]) or {"claims": []}

missing_context = context.get("missing_context", [])
contradictions = context.get("contradictions", [])
suggestions = context.get("suggested_supporting_facts", [])
suggested_claims = [item.get("suggested_claim") for item in suggestions if item.get("suggested_claim")]
suggested_claim_ids = {claim.get("claim_id") for claim in suggested_claims if claim.get("claim_id")}

aug_ver_df = pd.DataFrame(augmented.get("verifications", []))
verified_suggested_ids = set()
if not aug_ver_df.empty and suggested_claim_ids:
    verified_suggested_ids = set(
        aug_ver_df.loc[
            aug_ver_df["claim_id"].isin(suggested_claim_ids) & (aug_ver_df["status"] == "verified"),
            "claim_id",
        ]
    )

fact_sheet_rows = []
for section, entries in fact_sheet.get("sections", {}).items():
    for entry in entries:
        fact_sheet_rows.append(
            {
                "section": section,
                "text": entry.get("text", ""),
                "evidence_ids": entry.get("evidence_ids", []),
            }
        )

claim_by_id = {claim.get("claim_id"): claim for claim in claims_augmented.get("claims", [])}
verified_context_rows = []
for claim_id in sorted(verified_suggested_ids):
    claim = claim_by_id.get(claim_id, {})
    text_fragments = [str(v).lower() for v in (claim.get("predicate"), claim.get("value")) if v]
    matching_fact_rows = []
    for row in fact_sheet_rows:
        row_text = row["text"].lower()
        if text_fragments and any(fragment in row_text for fragment in text_fragments):
            matching_fact_rows.append(row)
    verified_context_rows.append(
        {
            "claim_id": claim_id,
            "type": claim.get("type"),
            "value": claim.get("value"),
            "verified": True,
            "fact_sheet_sections": ", ".join(sorted({row["section"] for row in matching_fact_rows})) or "not found",
            "fact_sheet_entries": len(matching_fact_rows),
        }
    )

context_summary = {
    "context_agent_enabled": ENABLE_CONTEXT_AGENT,
    "missing_context_items": len(missing_context),
    "suggested_ehr_claims": len(suggested_claims),
    "verified_suggested_claims": len(verified_suggested_ids),
    "verified_context_in_fact_sheet": sum(row["fact_sheet_entries"] > 0 for row in verified_context_rows),
    "contradictions": len(contradictions),
}
display(pd.DataFrame([context_summary]))

if not ENABLE_CONTEXT_AGENT:
    print("Context agent is OFF for this run; no EHR context will be added by stage 4.")
elif verified_suggested_ids:
    print(f"Context agent added {len(verified_suggested_ids)} EHR-backed missing-context claim(s) after S3 re-verification.")
else:
    print("Context agent ran, but no suggested EHR-backed context claims were verified and added.")

if missing_context:
    print("\nMissing EHR context flagged by context agent:")
    for item in missing_context:
        print("-", item)

if verified_context_rows:
    print("\nVerified context-agent additions:")
    display(pd.DataFrame(verified_context_rows))

if contradictions:
    print("\nContradictions:")
    for item in contradictions:
        print("-", item)

print("\nSuggested supporting facts:")
for item in suggestions:
    claim = item.get("suggested_claim") or {}
    claim_id = claim.get("claim_id", "")
    status = "verified" if claim_id in verified_suggested_ids else "not verified"
    print(f"- [{status}] {claim_id}: {item.get('description', item)}")

,missing_context,contradictions,suggested_supporting_facts
0,3,1,4


Missing context:
- Patient is critically ill with severe sepsis, septic shock, and acute respiratory failure, which provides critical context for the intubation and hemodynamic instability.
- Patient has a complex surgical history including right hemicolectomy and permanent ileostomy.
- Patient has comorbid Type II Diabetes and Atrial Fibrillation.

Contradictions:
- Claim C8 (COPD exacerbation) was marked unsupported because evidence E:cond:3 specifies 'Chronic obstructive asthma with exacerbation', though C22 was verified as COPD.

Suggested supporting facts:
- Severe sepsis and septic shock explain the need for intensive hemodynamic monitoring and vasopressors/sedation.
- Acute respiratory failure supports the clinical necessity of the mechanical ventilation mentioned in the notes.
- The presence of a permanent ileostomy is a significant surgical status for a clinician summary.
- Hemodialysis is a critical intervention for the acute renal failure and hyperkalemia described.


In [10]:
fact_sheet = read_json(artifact_paths["fact_sheet"]) or {"sections": {}}
section_rows = []
for section, entries in fact_sheet.get("sections", {}).items():
    section_rows.append({"section": section, "entries": len(entries)})

display(pd.DataFrame(section_rows))

for section, entries in fact_sheet.get("sections", {}).items():
    display(Markdown(f"### {section}"))
    if not entries:
        print("(none)")
    for entry in entries[:20]:
        ids = ", ".join(entry.get("evidence_ids", []))
        print(f"- {entry.get('text', '')} [{ids}]")
    if len(entries) > 20:
        print(f"... {len(entries) - 20} more")

,section,entries
0,hpi,0
1,active_problems,51
2,medications,8
3,labs,4
4,vitals,5
5,plan,3


### hpi

(none)


### active_problems

- Acute respiratory failure (2104-06-03) [E:cond:1]
- Acute vascular insufficiency of intestine (2104-06-03) [E:cond:2]
- Chronic obstructive asthma with (acute) exacerbation (2104-06-03) [E:cond:3]
- Hematoma complicating a procedure (2104-06-03) [E:cond:4]
- Cardiac complications, not elsewhere classified (2104-06-03) [E:cond:5]
- Acute kidney failure with lesion of tubular necrosis (2104-06-03) [E:cond:6]
- Severe sepsis (2104-06-03) [E:cond:7]
- Septic shock (2104-06-03) [E:cond:8]
- Unspecified septicemia (2104-06-03) [E:cond:9]
- Mixed acid-base balance disorder (2104-06-03) [E:cond:10]
- Aseptic necrosis of head and neck of femur (2104-06-03) [E:cond:11]
- Kidney replaced by transplant (2104-06-03) [E:cond:12]
- Hyperpotassemia (2104-06-03) [E:cond:13]
- Diagnosis (2104-06-03) [E:cond:14]
- Benign essential hypertension (2104-06-03) [E:cond:15]
- Diabetes mellitus without mention of complication, type II or unspecified type, not stated as uncontrolled (2104-06-03) [E:cond:16]
- 

...

- lab result: PM creat- 3.9 [E:note:row673718:23]
- lab result: BUN 72 [E:note:row673718:23]
- lab result: K-5.7 [E:note:row673718:23]
- lab result: elevated BUN/Creat [E:note:row673718:30]


### vitals

- lab result: NBP 193/113 [E:note:row673718:2]
- lab result: SBP 110 s to 170 s [E:note:row673718:3]
- lab result: HR 110 s to 120 s [E:note:row673718:3]
- lab result: SBP remained labile throughout shift ranging from 110 s to 170 [E:note:row673718:7]
- lab result: UOP about 30 to 50cc/hr [E:note:row673718:24]


### plan

- plan to: monitor patient hemodynamic status closely [E:note:row673718:8]
- plan to: Continue with propofol, fentanyl and hydralazine as ordered [E:note:row673718:9]
- plan to: increase hydralazine dose [E:note:row673718:10]


In [11]:
summary_md = read_text(artifact_paths["summary"])
revised_md = read_text(artifact_paths["summary_revised"])
patient_md = read_text(artifact_paths["patient_summary"])
summary_meta = read_json(artifact_paths["summary_meta"]) or {}
patient_meta = read_json(artifact_paths["patient_summary_meta"]) or {}

print("summary meta:")
display(pd.DataFrame([summary_meta]) if summary_meta else pd.DataFrame())

if patient_meta:
    print("patient summary meta:")
    display(pd.DataFrame([patient_meta]))

display(Markdown("## Clinician Summary"))
display(Markdown(summary_md or "_(missing)_"))

if revised_md:
    display(Markdown("## Revised Clinician Summary"))
    display(Markdown(revised_md))

if patient_md:
    display(Markdown("## Patient-Facing Summary"))
    display(Markdown(patient_md))

summary meta:


,note_chars,summary_chars,target_min_chars,target_max_chars,achieved_ratio,in_target_band
0,2295,1150,459,688,0.501089,False


patient summary meta:


,fk_grade,fk_words,fk_sentences,fk_syllables,target_grade_min,target_grade_max,in_target_band,chars
0,7.92,208,20,343,7.0,8.0,True,1828


## Clinician Summary

## HPI
_Not documented._

## Active Problems
Acute respiratory failure [E:cond:1], acute vascular insufficiency of intestine [E:cond:2], and COPD with acute exacerbation [E:cond:3][E:note:row673718:11]. Severe sepsis [E:cond:7], septic shock [E:cond:8], and unspecified septicemia [E:cond:9]. Acute kidney failure with tubular necrosis [E:cond:6][E:note:row673718:22] and hyperkalemia [E:cond:13][E:note:row673718:30]. History of renal transplant [E:cond:12], benign hypertension [E:cond:15][E:note:row673718:3], type II diabetes [E:cond:16], atrial fibrillation [E:cond:17], and prostate malignancy [E:cond:25].

## Medications
Fentanyl 100mcg/kg [E:note:row673718:4], propofol 90mcg/kg [E:note:row673718:4], IV hydralazine q6hr [E:note:row673718:5], Cellcept [E:note:row673718:26], and MDIs [E:note:row673718:15]. Prograf held [E:note:row673718:27].

## Labs
Creatinine 3.9, BUN 72, K 5.7 [E:note:row673718:23]. Elevated BUN/Creat [E:note:row673718:30].

## Plan
Monitor hemodynamic status closely [E:note:row673718:8]. Continue propofol, fentanyl, and hydralazine [E:note:row673718:9], with plan to increase hydralazine dose [E:note:row673718:10].


## Revised Clinician Summary

## HPI
_Not documented._

## Active Problems
Acute respiratory failure [E:cond:1], acute vascular insufficiency of intestine [E:cond:2], and chronic obstructive asthma with acute exacerbation [E:cond:3][E:note:row673718:11]. Severe sepsis [E:cond:7], septic shock [E:cond:8], and unspecified septicemia [E:cond:9]. Acute kidney failure with tubular necrosis [E:cond:6][E:note:row673718:22] and hyperkalemia [E:cond:13][E:note:row673718:30]. History of renal transplant [E:cond:12], benign hypertension [E:cond:15][E:note:row673718:3], type II diabetes [E:cond:16], atrial fibrillation [E:cond:17], and prostate malignancy [E:cond:25].

## Medications
Fentanyl 100mcg/kg [E:note:row673718:4], propofol 90mcg/kg [E:note:row673718:4], IV hydralazine q6hr [E:note:row673718:5], Cellcept [E:note:row673718:26], and MDIs [E:note:row673718:15]. Prograf held [E:note:row673718:27].

## Labs
Creatinine 3.9, BUN 72, K 5.7 [E:note:row673718:23]. Elevated BUN/Creat [E:note:row673718:30].

## Plan
Monitor hemodynamic status closely [E:note:row673718:8]. Continue propofol, fentanyl, and hydralazine [E:note:row673718:9], with plan to increase hydralazine dose [E:note:row673718:10].


## Patient-Facing Summary

## What happened
You had a very serious stay in the hospital. You needed a breathing machine for at least 96 hours [E:proc:10]. You had several surgeries on your intestines [E:proc:1][E:proc:2][E:proc:4]. You also had a permanent ileostomy (a surgical opening in the belly for waste) [E:proc:5]. You received dialysis for your kidneys [E:proc:11] and a kidney biopsy [E:proc:9]. We also used a procedure to fix your irregular heartbeat [E:proc:15].

## Your conditions
You were treated for severe sepsis and septic shock [E:cond:7][E:cond:8]. You had acute kidney failure [E:cond:6] and acute respiratory failure [E:cond:1]. You also had a lack of blood flow to your intestine [E:cond:2]. Other conditions include:
* High blood pressure [E:cond:15]
* Type 2 diabetes [E:cond:16]
* Irregular heartbeat (atrial fibrillation) [E:cond:17]
* Asthma with a sudden worsening [E:cond:3]
* A kidney transplant [E:cond:12]
* Low white blood cell count (neutropenia) [E:cond:18]
* Bone loss (osteoporosis) [E:cond:21]
* History of prostate cancer [E:cond:25]

## Your medicines
You took fentanyl and propofol [E:note:row673718:4]. You received hydralazine for blood pressure [E:note:row673718:5]. You used Cellcept [E:note:row673718:26]. You received a dose of K-excelate [E:note:row673718:29] and used inhalers (MDIs) [E:note:row673718:15]. Your Prograf was held [E:note:row673718:27].

## Your test results
Your blood pressure was high at 193/113 [E:note:row673718:2]. Your potassium was high at 5.7 [E:note:row673718:23]. Your BUN was 72 and creatinine was 3.9 [E:note:row673718:23].

## What to do next
We will keep a close watch on your heart and blood pressure [E:note:row673718:8]. You will continue taking propofol, fentanyl, and hydralazine [E:note:row673718:9]. We plan to increase your dose of hydralazine [E:note:row673718:10].


In [12]:
check_report = read_json(artifact_paths["check_report"]) or {}
check_revised = read_json(artifact_paths["check_report_revised"]) or {}
review = read_json(artifact_paths["review"]) or {}

check_rows = []
for name, report in [("original", check_report), ("revised", check_revised)]:
    if report:
        check_rows.append({
            "summary": name,
            "passed": report.get("passed"),
            "sentence_count": report.get("sentence_count"),
            "violations": len(report.get("violations", [])),
            "cited_evidence_ids": len(report.get("cited_evidence_ids", [])),
        })

display(pd.DataFrame(check_rows))

if check_report.get("violations"):
    display(Markdown("### Deterministic Check Violations"))
    display(pd.DataFrame(check_report["violations"]))

concerns = review.get("concerns", [])
revisions = review.get("recommended_revisions", [])
print("review concerns:", len(concerns))
if concerns:
    display(pd.DataFrame(concerns))
print("recommended revisions:", len(revisions))
if revisions:
    display(pd.DataFrame(revisions))

,summary,passed,sentence_count,violations,cited_evidence_ids
0,original,True,11,0,26
1,revised,True,11,0,26


review concerns: 1


,sentence,severity,reason
0,"Acute respiratory failure [E:cond:1], acute va...",medium,The fact sheet specifies 'Chronic obstructive ...


recommended revisions: 1


,original,suggested,reason
0,"Acute respiratory failure [E:cond:1], acute va...","Acute respiratory failure [E:cond:1], acute va...",Corrects the diagnosis from COPD to chronic ob...


In [13]:
# Reference-based metrics for this single case.
# Use the revised summary if it exists and passed deterministic re-check; otherwise use the original.
audit = read_json(artifact_paths["audit"]) or {}
use_revised = bool(revised_md and audit.get("revised_check_passed"))
prediction = revised_md if use_revised else summary_md
reference = mimic.real_case_reference_summary(case)
gold_entities = mimic.real_case_gold_entities(case)

metric_row = {
    "case_id": case.case_id,
    "prediction_source": "revised" if use_revised else "original",
}
if prediction.strip():
    metric_row.update(metrics.rouge_scores(prediction, reference))
    metric_row.update(metrics.entity_recall_precision(
        prediction=prediction,
        reference=reference,
        gold_entities=gold_entities,
    ))
    metric_row.update(metrics.flesch_kincaid_grade(prediction))
else:
    print("No prediction text to score.")

pd.DataFrame([metric_row])

2026-05-03 17:23:06,501 INFO absl: Using default tokenizer.


,case_id,prediction_source,rouge1_f,rouge2_f,rougeL_f,entity_precision,entity_recall,entity_f1,entity_tp,entity_fp,entity_fn,entity_gold,fk_grade,fk_words,fk_sentences,fk_syllables,fk_in_target_band
0,mimic-81475-101662-673718,revised,0.235294,0.108747,0.174118,1.0,0.272727,0.428571,12,0,32,44,13.72,88,10,193,False


In [14]:
# Optional: BERTScore is slower because it loads roberta-large the first time.
RUN_BERTSCORE = False

if RUN_BERTSCORE and prediction.strip():
    pd.DataFrame([
        metrics.bertscore_scores(prediction, reference, model_type=BERTSCORE_MODEL)
        | {"case_id": case.case_id}
    ])
else:
    print("Set RUN_BERTSCORE = True to compute BERTScore for this case.")

Set RUN_BERTSCORE = True to compute BERTScore for this case.


In [15]:
display(Markdown("## Reference Summary"))
display(Markdown(reference))

print("Gold entities:")
for ent in sorted(gold_entities):
    print("-", ent)

## Reference Summary

## HPI
61.0-year-old male.
## Active Problems
- Dyspnea.
- Wheezing.
- Hypertensive disease.
- Chronic Obstructive Airway Disease.
- Chronic obstructive pulmonary disease of horses.
- Kidney Failure.
## EHR Diagnoses
- Acute respiratory failure.
- Acute vascular insufficiency of intestine.
- Chronic obstructive asthma with (acute) exacerbation.
- Hematoma complicating a procedure.
- Cardiac complications, not elsewhere classified.
- Acute kidney failure with lesion of tubular necrosis.
- Severe sepsis.
- Septic shock.
- Unspecified septicemia.
- Mixed acid-base balance disorder.
- Aseptic necrosis of head and neck of femur.
- Kidney replaced by transplant.
- Hyperpotassemia.
- None.
- Benign essential hypertension.
- Diabetes mellitus without mention of complication, type II or unspecified type, not stated as uncontrolled.
- Atrial fibrillation.
- Neutropenia, unspecified.
- Other specified disorders of intestine.
- Esophageal reflux.
- Osteoporosis, unspecified.
- Hip joint replacement.
- Personal history of irradiation, presenting hazards to health.
- Personal history of noncompliance with medical treatment, presenting hazards to health.
- Personal history of malignant neoplasm of prostate.
## Procedures
- Open and other right hemicolectomy.
- Other partial resection of small intestine.
- Other partial resection of small intestine.
- Other and unspecified partial excision of large intestine.
- Other permanent ileostomy.
- Excision or destruction of peritoneal tissue.
- Delayed closure of granulating abdominal wound.
- Other open incisional hernia repair with graft or prosthesis.
- Closed [percutaneous] [needle] biopsy of kidney.
- Continuous invasive mechanical ventilation for 96 consecutive hours or more.
- Hemodialysis.
- Parenteral infusion of concentrated nutritional substances.
- Enteral infusion of concentrated nutritional substances.
- Venous catheterization, not elsewhere classified.
- Atrial cardioversion.
## Disposition
Patient expired during this admission.


Gold entities:
- acute kidney failure with lesion of tubular necrosis
- acute respiratory failure
- acute vascular insufficiency of intestine
- aseptic necrosis of head and neck of femur
- atrial cardioversion
- atrial fibrillation
- benign essential hypertension
- cardiac complications, not elsewhere classified
- chronic obstructive airway disease
- chronic obstructive asthma with (acute) exacerbation
- chronic obstructive pulmonary disease of horses
- closed [percutaneous] [needle] biopsy of kidney
- continuous invasive mechanical ventilation for 96 consecutive hours or more
- delayed closure of granulating abdominal wound
- diabetes mellitus without mention of complication, type ii or unspecified type, not stated as uncontrolled
- dyspnea
- enteral infusion of concentrated nutritional substances
- esophageal reflux
- excision or destruction of peritoneal tissue
- hematoma complicating a procedure
- hemodialysis
- hip joint replacement
- hyperpotassemia
- hypertensive disease
- kidne